## 필수 · 기본 문제 1. 뉴스 샘플 schema 감사기

### 문제 배경

빈 텍스트, 중복 ID, 허용되지 않은 label을 tokenization 이후에 찾으면 원문과 오류를 연결하기 어렵습니다. 원본 행 단계에서 오류 유형과 위치를 반환하는 감사기를 만듭니다.

### 시작 코드

```python
rows = [
    {"id": "n001", "text": "금리 인상 가능성에 시장 긴장", "label": "economy"},
    {"id": "n002", "text": "대표팀 결승 진출", "label": "sports"},
    {"id": "n002", "text": "  ", "label": "unknown"},
]
ALLOWED_LABELS = {"economy", "sports", "tech"}

def audit_rows(rows, allowed_labels):
    raise NotImplementedError
```

### 수행 요구사항

1. 필수 key `id/text/label` 누락을 찾으세요.
2. 공백만 있는 텍스트, 중복 ID, 허용되지 않은 label의 행 index를 기록하세요.
3. `valid`와 label별 개수를 반환하세요.
4. 오류가 있어도 가능한 모든 검사를 끝내고 한 report로 반환하세요.

### 제출 결과

- 전체 audit report
- 오류 3종과 해당 index
- Tokenization 전에 감사해야 하는 이유 2문장
- `기본 문제 1 자동 검증: PASS`

### 자동 검증

```python
report = audit_rows(rows, ALLOWED_LABELS)
assert report["duplicate_ids"] == ["n002"]
assert report["blank_text_rows"] == [2]
assert report["invalid_label_rows"] == [2]
assert report["valid"] is False
print("기본 문제 1 자동 검증: PASS")

    **자주 하는 실수**
    
    - `if row["text"]`만 검사해 공백 문자열을 놓칩니다.
    - 중복 행이 아니라 중복 ID의 고유 목록을 요구했는데 같은 ID를 여러 번 반환합니다.
    - 허용되지 않은 label까지 class 분포에 포함합니다.

검사 순서 · 한 행에서 필수 key와 빈 값을 확인한 뒤, 전체 행을 모아 ID 중복과 label 분포를 계산합니다. 오류가 여러 개 겹친 행도 한 번에 보고할 수 있도록 즉시 중단하지 않고 목록에 모읍니다.

In [1]:
from collections import Counter

rows = [
    {"id": "n001", "text": "금리 인상 가능성에 시장 긴장", "label": "economy"},
    {"id": "n002", "text": "대표팀 결승 진출", "label": "sports"},
    {"id": "n002", "text": "  ", "label": "unknown"},
]
ALLOWED_LABELS = {"economy", "sports", "tech"}

def audit_rows(rows, allowed_labels):
    # 서로 다른 오류를 한 번에 보여 주기 위해 종류별 index 목록을 만듭니다.
    required = {"id", "text", "label"}
    missing_key_rows, blank_text_rows, invalid_label_rows = [], [], []
    ids, labels = [], []
    for index, row in enumerate(rows):
        missing = sorted(required - row.keys())
        if missing:
            missing_key_rows.append({"index": index, "keys": missing})
        row_id = row.get("id")
        if row_id:
            ids.append(row_id)
        if not isinstance(row.get("text"), str) or not row.get("text", "").strip():
            blank_text_rows.append(index)
        label = row.get("label")
        if label not in allowed_labels:
            invalid_label_rows.append(index)
        elif label is not None:
            labels.append(label)
    # 행 단위 검사가 끝난 뒤 전체 ID 중복과 label 분포를 집계합니다.
    id_counts = Counter(ids)
    duplicate_ids = sorted(row_id for row_id, count in id_counts.items() if count > 1)
    valid = not (missing_key_rows or blank_text_rows or invalid_label_rows or duplicate_ids)
    return {
        "row_count": len(rows), "missing_key_rows": missing_key_rows,
        "blank_text_rows": blank_text_rows, "invalid_label_rows": invalid_label_rows,
        "duplicate_ids": duplicate_ids, "label_counts": dict(sorted(Counter(labels).items())),
        "valid": valid,
    }

report = audit_rows(rows, ALLOWED_LABELS)
print(report)
assert report["duplicate_ids"] == ["n002"]
assert report["blank_text_rows"] == [2]
assert report["invalid_label_rows"] == [2]
assert report["valid"] is False
print("기본 문제 1 자동 검증: PASS")

{'row_count': 3, 'missing_key_rows': [], 'blank_text_rows': [2], 'invalid_label_rows': [2], 'duplicate_ids': ['n002'], 'label_counts': {'economy': 1, 'sports': 1}, 'valid': False}
기본 문제 1 자동 검증: PASS


  상세 해설 즉시 예외를 내지 않고 모든 행을
  검사하면 한 번의 수정 주기에 여러 오류를 고칠 수 있습니다. Label count에서는 허용된 label만 세어 오류 label이 정상 분포를 왜곡하지 않게 했습니다.

## 필수 · 기본 문제 2. Toy subword tokenizer와 ID 왕복

### 문제 배경

실제 tokenizer 내부는 복잡합니다. 먼저 “긴 token을 우선 선택하는 greedy subword 분할”을 작은 vocabulary로 구현해 token과 ID의 관계, `[CLS]/[SEP]/[UNK]` 역할을 분리합니다.

### 시작 코드

```python
vocab = {"[PAD]": 0, "[UNK]": 1, "[CLS]": 2, "[SEP]": 3,
         "인공": 4, "##지능": 5, "연구": 6, "##소": 7, "출범": 8}

def encode_words(words, vocab):
    raise NotImplementedError
```

### 수행 요구사항

1. 각 word의 첫 조각은 원문 표기, 뒤 조각은 `##` prefix 후보에서 찾으세요.
2. 가능한 후보 중 가장 긴 문자열을 우선하세요.
3. 한 word를 끝까지 분할하지 못하면 그 word 전체를 `[UNK]` 하나로 바꾸세요.
4. 전체 앞뒤에 `[CLS]`, `[SEP]`를 넣고 token과 ID를 모두 반환하세요.

### 제출 결과

- `['인공지능', '연구소', '출범']`의 token·ID
- `[UNK]`가 생기는 입력 한 개
- `기본 문제 2 자동 검증: PASS`

### 자동 검증

```python
tokens, ids = encode_words(["인공지능", "연구소", "출범"], vocab)
assert tokens == ["[CLS]", "인공", "##지능", "연구", "##소", "출범", "[SEP]"]
assert ids == [vocab[token] for token in tokens]
print("기본 문제 2 자동 검증: PASS")
    

    
    **자주 하는 실수**
    
    - 뒤 조각에 `##`를 붙이지 않아 vocabulary lookup이 실패합니다.
    - 가장 짧은 조각부터 골라 불필요하게 sequence를 늘립니다.
    - Word 일부를 분할한 뒤 실패했는데 앞 조각과 `[UNK]`를 함께 남깁니다.

In [2]:
vocab = {"[PAD]": 0, "[UNK]": 1, "[CLS]": 2, "[SEP]": 3,
         "인공": 4, "##지능": 5, "연구": 6, "##소": 7, "출범": 8}

def split_word(word, vocab):
    pieces, start = [], 0
    while start < len(word):
        best = None
        # 현재 위치에서 가능한 가장 긴 조각부터 검사합니다.
        for end in range(len(word), start, -1):
            raw = word[start:end]
            candidate = raw if start == 0 else "##" + raw
            if candidate in vocab:
                best = (candidate, end)
                break
        if best is None:
            # 일부 조각만 남기지 않고 word 전체를 UNK 하나로 처리합니다.
            return ["[UNK]"]
        token, start = best
        pieces.append(token)
    return pieces

def encode_words(words, vocab):
    tokens = ["[CLS]"]
    for word in words:
        tokens.extend(split_word(word, vocab))
    tokens.append("[SEP]")
    ids = [vocab[token] for token in tokens]
    return tokens, ids

tokens, ids = encode_words(["인공지능", "연구소", "출범"], vocab)
unknown_tokens, _ = encode_words(["미등록"], vocab)
print("tokens:", tokens)
print("ids:", ids)
print("unknown:", unknown_tokens)
assert tokens == ["[CLS]", "인공", "##지능", "연구", "##소", "출범", "[SEP]"]
assert ids == [vocab[token] for token in tokens]
assert "[UNK]" in unknown_tokens
print("기본 문제 2 자동 검증: PASS")

tokens: ['[CLS]', '인공', '##지능', '연구', '##소', '출범', '[SEP]']
ids: [2, 4, 5, 6, 7, 8, 3]
unknown: ['[CLS]', '[UNK]', '[SEP]']
기본 문제 2 자동 검증: PASS


**풀이 핵심** · 각 시작 위치에서 가장 긴 등록 조각부터 거꾸로 탐색하는 greedy longest-match를 사용합니다. 중간부터 시작하는 조각에만 `##`를 붙이고, 끝까지 분해하지 못하면 단어 전체를 `[UNK]` 하나로 처리합니다.

  이 구현은 WordPiece의 핵심 직관만 단순화한 것입니다. 실제 tokenizer는 정규화, pre-tokenization, 언어별 규칙과 학습된 vocabulary를 함께 사용하므로 이 결과를 KoELECTRA의 실제 분할이라고 해석하지 않습니다.